In [66]:
import GEOparse, scanpy as sc, urllib, os, gzip, pandas as pd, tempfile, shutil

In [67]:
gse = GEOparse.get_GEO(geo='GSE274986', destdir='./')

14-Oct-2026 14:51:39 DEBUG utils - Directory ./ already exists. Skipping.
14-Oct-2026 14:51:39 INFO GEOparse - File already exist: using local version.
14-Oct-2026 14:51:39 INFO GEOparse - Parsing ./GSE274986_family.soft.gz: 
14-Oct-2026 14:51:39 DEBUG GEOparse - DATABASE: GeoMiame
14-Oct-2026 14:51:39 DEBUG GEOparse - SERIES: GSE274986
14-Oct-2026 14:51:39 DEBUG GEOparse - PLATFORM: GPL34281
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820702
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820703
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820704
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820705
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820706
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820707
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820708
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820709
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820710
14-Oct-2026 14:51:39 DEBUG GEOparse - SAMPLE: GSM8820711
14-Oct-2026 14:51:39 DEBUG GEOpars

In [70]:
#Test list so I can run code without having to do the entire sample list
test_keys_list = list(gse.gsms.keys())[:3]
test_dict = {k: gse.gsms[k] for k in test_keys_list}


In [72]:
file_dict = {}

#Replace with gse.gsms.items()
for sample, data in test_dict.items():
    matrix_link_list = data.metadata.get('supplementary_file_4')
    matrix_first_item = matrix_link_list[0]
    matrix_file_name = os.path.basename(matrix_first_item)

    #Note: urlretrieve(matrix_first_item, matrix_file_name) 
    #will return a list so you need to retrieve the first item
    matrix_list = urllib.request.urlretrieve(matrix_first_item, matrix_file_name)

    features_link_list = data.metadata.get('supplementary_file_3')
    features_first_item =  features_link_list[0]
    features_file_name = os.path.basename(features_first_item)
    features_list = urllib.request.urlretrieve(features_first_item, features_file_name)

    barcodes_link_list = data.metadata.get('supplementary_file_2')
    barcodes_first_item = barcodes_link_list[0]
    barcodes_file_name = os.path.basename(barcodes_first_item)
    barcodes_list = urllib.request.urlretrieve(barcodes_first_item, barcodes_file_name)
    
    file_mappings = {
        'matrix.mtx.gz' : matrix_list[0],
        'features.tsv.gz' : features_list[0],
        'barcodes.tsv.gz' : barcodes_list[0] 
    }
    print(f'Downloaded matrix, features, barcodes for {sample}')

    file_dict[sample] = file_mappings
    

Downloaded matrix, features, barcodes for GSM8820702
Downloaded matrix, features, barcodes for GSM8820703
Downloaded matrix, features, barcodes for GSM8820704


In [78]:

adata_dict = {}

source_dir = os.getcwd()

for sample_id, sample_files in file_dict.items():
    with tempfile.TemporaryDirectory() as tmpdir:
            for standard_name, custom_name in sample_files.items():
                src_path = os.path.join(source_dir, custom_name)
                dst_path = os.path.join(tmpdir, standard_name)
            
            # Use symlink if you want to save space/time, or copy if files are elsewhere
                x = shutil.copy(src_path, dst_path)

            adata = sc.read_10x_mtx(
                    path = tmpdir,
                    make_unique=True,
                    var_names='gene_symbols'
                )

            adata_dict[sample_id] = adata
            

In [79]:
adata_dict.items()

dict_items([('GSM8820702', AnnData object with n_obs × n_vars = 10405 × 36601
    var: 'gene_ids', 'feature_types'), ('GSM8820703', AnnData object with n_obs × n_vars = 10472 × 36601
    var: 'gene_ids', 'feature_types'), ('GSM8820704', AnnData object with n_obs × n_vars = 3656 × 36601
    var: 'gene_ids', 'feature_types')])

In [81]:
print(os.path.abspath(''))

C:\Users\josem\Reeck Lab\extracted_data
